## Faiyad Mahabub Part A (Q4 and Q5)

*Q4: Sentence Pobabilities using Bigram Language Models - Python Implementation* 

**Imports**

In [6]:
import nltk
from nltk.lm import MLE, Laplace
import numpy as np
from nltk.lm.preprocessing import padded_everygram_pipeline

**Loading Data Into Workspace**

In [7]:
# Load file
file_path = r"C:\Asia Pacific University\All Module Notes\Semister-5\Text Analysis & Sentimental Analysis\Assignment\CT107-3-3-TXSA - Group Assignment\Data\Data_3.txt"

with open(file_path, 'r') as f:
    lines = f.readlines()
print(len(lines))

9


**Preparing Training Data**

***Separating Training Corpus And Test Sentence***

In [8]:
# Separate corpus and test sentence
corpus = [lines[2].strip(), lines[3].strip(), lines[4].strip()]  # Training sentences
test_sentence = lines[8].strip()  # Test sentence

print("Corpus:", corpus)
print("\nTest:", test_sentence)

Corpus: ['<s> He read a book </s>', '<s> I read a different book </s>', '<s> He read a book by Danielle </s>']

Test: <s> I read a different book by Danielle </s>


***Tokenization for both training and test sentences***

In [9]:
# Tokenize by splitting on spaces
tokenized_text = [sent.split() for sent in corpus]
test_tokens = test_sentence.split()

print("Tokenized corpus:\n", tokenized_text)
print("\nTokenized test:\n", test_tokens)
print()

Tokenized corpus:
 [['<s>', 'He', 'read', 'a', 'book', '</s>'], ['<s>', 'I', 'read', 'a', 'different', 'book', '</s>'], ['<s>', 'He', 'read', 'a', 'book', 'by', 'Danielle', '</s>']]

Tokenized test:
 ['<s>', 'I', 'read', 'a', 'different', 'book', 'by', 'Danielle', '</s>']



***Stripping Sentence Markers*** 

In [10]:
# Strip existing markers in training corpus
clean_tokenized = [[w for w in sent if w not in ['<s>', '</s>']] 
                                for sent in tokenized_text]
print("Cleaned tokenized corpus:\n", clean_tokenized)

Cleaned tokenized corpus:
 [['He', 'read', 'a', 'book'], ['I', 'read', 'a', 'different', 'book'], ['He', 'read', 'a', 'book', 'by', 'Danielle']]


***All Bigrams In Test Sentence***

In [11]:
test_bigrams = list(nltk.bigrams(test_tokens))
print("\nTest bigrams:\n", test_bigrams)


Test bigrams:
 [('<s>', 'I'), ('I', 'read'), ('read', 'a'), ('a', 'different'), ('different', 'book'), ('book', 'by'), ('by', 'Danielle'), ('Danielle', '</s>')]


**Model Training (unsmoothed)**

In [12]:
#using padded everygram pipeline to create  unigrams + bigrams and adds markers
train_data, vocab_data = padded_everygram_pipeline(2, clean_tokenized)

model_mle = MLE(2)
model_mle.fit(train_data, vocab_data)

print(f"Vocabulary size: {len(model_mle.vocab)}")
print("MLE Model N-grams:", model_mle.counts)

Vocabulary size: 11
MLE Model N-grams: <NgramCounter with 2 ngram orders and 39 ngrams>


***MLE Probabilities for All Bigrams***

In [13]:
# Calculate unsmoothed probabilities
print("Unsmoothed probabilities:")
mle_probs = []

for w1, w2 in test_bigrams:
    prob = model_mle.score(w2, [w1])
    mle_probs.append(prob)
    print(f"  P({w2}|{w1}) = {prob:.4f}")

Unsmoothed probabilities:
  P(I|<s>) = 0.3333
  P(read|I) = 1.0000
  P(a|read) = 1.0000
  P(different|a) = 0.3333
  P(book|different) = 1.0000
  P(by|book) = 0.3333
  P(Danielle|by) = 1.0000
  P(</s>|Danielle) = 1.0000


***Overall Sentence Probability***

In [14]:
sentence_prob_mle = np.prod(mle_probs)

print(f"Sentence probability = {sentence_prob_mle:.6f}")
print(f"                     = {sentence_prob_mle:.2e}")

Sentence probability = 0.037037
                     = 3.70e-02


**Training Laplace Model (Smoothed Bigram Model)**

In [15]:
train_data, vocab_data = padded_everygram_pipeline(2, clean_tokenized)
model_laplace = Laplace(2)
model_laplace.fit(train_data, vocab_data)
print(f"Laplace model trained (V = {len(model_laplace.vocab)})")

Laplace model trained (V = 11)


***Laplace Probabilities for All Bigrams***

In [16]:
# Calculate smoothed probabilities
print("Smoothed probabilities:")
laplace_probs = []

for w1, w2 in test_bigrams:
    prob = model_laplace.score(w2, [w1])
    laplace_probs.append(prob)
    print(f"  P({w2}|{w1}) = {prob:.4f}")

Smoothed probabilities:
  P(I|<s>) = 0.1429
  P(read|I) = 0.1667
  P(a|read) = 0.2857
  P(different|a) = 0.1429
  P(book|different) = 0.1667
  P(by|book) = 0.1429
  P(Danielle|by) = 0.1667
  P(</s>|Danielle) = 0.1667


***Overall Sentence Probability using Laplace***

In [17]:
# Calculate sentence probability
sentence_prob_laplace = np.prod(laplace_probs)

print(f"Sentence probability = {sentence_prob_laplace:.10f}")
print(f"                     = {sentence_prob_laplace:.2e}")

Sentence probability = 0.0000006427
                     = 6.43e-07


*Q5: Alternative Approach Implementation* 

**Imports**

In [24]:
import stanza
import string
import time
import pandas as pd
from collections import Counter
from IPython.display import HTML

**Loading Text Corpus**

In [25]:
with open(r"C:\Asia Pacific University\All Module Notes\Semister-5\Text Analysis & Sentimental Analysis\Assignment\CT107-3-3-TXSA - Group Assignment\Data\Data_1.txt") as f:
    text = f.read()

print("Corpus:")
print(text)

Corpus:
Classification is the task of choosing the correct class label for a given input. In basic
classification tasks, each input is considered in isolation from all other inputs, and the set of labels is defined in advance. The basic classification task has a number of interesting variants. For example, in multiclass classification, each instance may be assigned multiple labels; in open-class classification, the set of labels is not defined in advance; and in sequence classification, a list of inputs are jointly classified.


**Tokenization using Stanza**

In [26]:
# Tokenize using Stanza
nlp = stanza.Pipeline(lang='en', processors='tokenize', verbose=False)

start = time.time()
doc = nlp(text)
end = time.time()

stanza_tokens = [token.text for sentence in doc.sentences for token in sentence.tokens]
execution_time_ms = (end - start) * 1000

print("Stanza Tokens:\n")
for i, token in enumerate(stanza_tokens):
    print(f"{i+1:>3}. {token}")
print(f"Execution Time: {execution_time_ms:.2f} ms")

Stanza Tokens:

  1. Classification
  2. is
  3. the
  4. task
  5. of
  6. choosing
  7. the
  8. correct
  9. class
 10. label
 11. for
 12. a
 13. given
 14. input
 15. .
 16. In
 17. basic
 18. classification
 19. tasks
 20. ,
 21. each
 22. input
 23. is
 24. considered
 25. in
 26. isolation
 27. from
 28. all
 29. other
 30. inputs
 31. ,
 32. and
 33. the
 34. set
 35. of
 36. labels
 37. is
 38. defined
 39. in
 40. advance
 41. .
 42. The
 43. basic
 44. classification
 45. task
 46. has
 47. a
 48. number
 49. of
 50. interesting
 51. variants
 52. .
 53. For
 54. example
 55. ,
 56. in
 57. multiclass
 58. classification
 59. ,
 60. each
 61. instance
 62. may
 63. be
 64. assigned
 65. multiple
 66. labels
 67. ;
 68. in
 69. open
 70. -
 71. class
 72. classification
 73. ,
 74. the
 75. set
 76. of
 77. labels
 78. is
 79. not
 80. defined
 81. in
 82. advance
 83. ;
 84. and
 85. in
 86. sequence
 87. classification
 88. ,
 89. a
 90. list
 91. of
 92. inputs
 93. are
 

***Metrics For Stanza***

In [ ]:
# Metric 1: Total Tokens
total_tokens = len(stanza_tokens)

# Metric 2: Tokens with Punctuation Attached
punct_attached = sum(
    1 for t in stanza_tokens
    if any(p in t for p in string.punctuation) and not all(p in string.punctuation for p in t)
)

# ---------- Metric 3: Unique Tokens ----------
# Metric 3a: Unique Tokens (Case-Sensitive) 
unique_case_sensitive = len(set(stanza_tokens))

# Metric 3b: Unique Tokens (Lowercased)
unique_lowercased = len(set(t.lower() for t in stanza_tokens))
vocab_inflation   = unique_case_sensitive - unique_lowercased

# Metric 3c: Case-Sensitive Duplicate Pairs
token_counter = Counter(stanza_tokens)
case_pairs = [
    (t, t.lower()) for t in token_counter
    if t[0].isupper() and t.lower() in token_counter
]

# Metric 4: Hyphenated Words Split Incorrectly
hyphen_words = [word for word in text.split() if '-' in word]
hyphen_split = sum(
    1 for hw in hyphen_words
    if hw not in stanza_tokens
)


# Metric 6: Punctuation Type Breakdown
punct_types = {
    'Periods (.)':    [t for t in stanza_tokens if t == '.'],
    'Commas (,)':     [t for t in stanza_tokens if t == ','],
    'Semicolons (;)': [t for t in stanza_tokens if t == ';'],
    'Hyphens (-)':    [t for t in stanza_tokens if t == '-'],
}



***Full Summary Output of the Metrics***

In [ ]:
from IPython.display import HTML

metrics_data = {
    "Metric": [
        "Total Tokens",
        "Tokens with Punctuation Attached",
        "Unique Tokens (Case-Sensitive)",
        "Unique Tokens (Lowercased)",
        "Vocabulary Inflation Due to Case",
        "Case-Sensitive Duplicate Pairs",
        "Hyphenated Words in Corpus",
        "Hyphenated Words Split Incorrectly",
        "Execution Time (ms)"
    ],
    "Value": [
        total_tokens,
        punct_attached,
        unique_case_sensitive,
        unique_lowercased,
        vocab_inflation,
        ", ".join([f"('{a}' / '{b}')" for a, b in case_pairs]),
        len(hyphen_words),
        hyphen_split,
        f"{execution_time_ms:.2f} ms"
    ]
}

df_metrics = pd.DataFrame(metrics_data)

html = df_metrics.to_html(index=False)

display(HTML(f"""
<div style="background:#fff; padding:20px; border-radius:8px; width:fit-content; box-shadow:0 2px 8px rgba(0,0,0,0.15)">
    <h4 style="color:#1f4e79; margin-bottom:12px">Stanza Tokenization Metrics</h4>
    <style>
        table {{border-collapse:collapse; width:600px; font-size:13px}}
        th    {{background:#1f4e79; color:#fff; padding:10px 20px; text-align:center}}
        td    {{padding:9px 20px; border:1px solid #b8cce4; color:#000}}
        td:first-child  {{text-align:left}}
        td:last-child   {{text-align:center; font-weight:bold}}
        tr:nth-child(even) td {{background:#dce6f1}}
        tr:nth-child(odd)  td {{background:#ffffff}}
        tr:hover td {{background:#bdd7ee}}
    </style>
    {html}
</div>
"""))

Metric,Value
Total Tokens,96
Tokens with Punctuation Attached,0
Unique Tokens (Case-Sensitive),54
Unique Tokens (Lowercased),50
Vocabulary Inflation Due to Case,4
Case-Sensitive Duplicate Pairs,"('Classification' / 'classification'), ('In' / 'in'), ('The' / 'the'), ('For' / 'for')"
Hyphenated Words in Corpus,1
Hyphenated Words Split Incorrectly,1
Execution Time (ms),134.37 ms


: 